# Child Nutrition Model Pipeline

This notebook trains the Decision Tree model, extracts the label encoders to a JSON file, and exports the model to a secure ONNX format. This creates a production-ready artifact that can easily be ingested by the LangGraph agentic framework.

In [ ]:
!pip install pandas scikit-learn skl2onnx onnxruntime

In [ ]:
import pandas as pd
import json
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as rt

# 1. Load the dataset (Assuming the file is in a 'data' folder or same directory)
try:
    df = pd.read_csv('data/child_nutrition_data.csv')
except FileNotFoundError:
    df = pd.read_csv('child_nutrition_data.csv')

print(f"Dataset loaded with {df.shape[0]} rows and {df.shape[1]} columns.")

In [ ]:
# 2. Initialize encoders and extract mappings
encoders = {
    'Gender': LabelEncoder(),
    'Has_Regular_Meals': LabelEncoder(),
    'Eats_Fruits_Veggies': LabelEncoder(),
    'Clean_Drinking_Water': LabelEncoder(),
    'Nutrition_Status': LabelEncoder()
}

label_mappings = {}

for col, le in encoders.items():
    df[col] = le.fit_transform(df[col])
    mapping = {str(class_name): int(encoded_val) for encoded_val, class_name in enumerate(le.classes_)}
    label_mappings[col] = mapping

# Save the mappings to a standard JSON file
with open('label_encoders.json', 'w') as f:
    json.dump(label_mappings, f, indent=4)

print("✅ Label encodings saved successfully to 'label_encoders.json'")

In [ ]:
# 3. Prepare Features and Target for ONNX (float32 is required for tensor matching)
X = df.drop('Nutrition_Status', axis=1).astype('float32')
y = df['Nutrition_Status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the Decision Tree
clf = DecisionTreeClassifier(random_state=42, max_depth=5)
clf.fit(X_train, y_train)

accuracy = clf.score(X_test, y_test)
print(f"✅ Model trained successfully! Test Accuracy: {accuracy:.2f}")

In [ ]:
# 4. Convert and Export to ONNX
initial_type = [('float_input', FloatTensorType([None, X_train.shape[1]]))]
onnx_model = convert_sklearn(clf, initial_types=initial_type)

with open('child_nutrition.onnx', 'wb') as f:
    f.write(onnx_model.SerializeToString())

print("✅ Model successfully exported to 'child_nutrition.onnx'")

In [ ]:
# 5. Verify ONNX Inference (Sanity Check)
sess = rt.InferenceSession('child_nutrition.onnx')
input_name = sess.get_inputs()[0].name
label_name = sess.get_outputs()[0].name

sample_input = X_test.iloc[0].values.reshape(1, -1).astype(np.float32)

onnx_prediction = sess.run([label_name], {input_name: sample_input})[0]
sklearn_prediction = clf.predict(sample_input)

print(f"Sklearn Prediction: {sklearn_prediction[0]}")
print(f"ONNX Prediction:    {onnx_prediction[0]}")
assert onnx_prediction[0] == sklearn_prediction[0], "Predictions do not match!"
print("✅ Verification passed: ONNX model behaves identically to the Sklearn model.")